## Cargar y verificar el dataset en Google Colab

Primero, vamos a cargar todos los datos al entorno de ejecución de Google Colab. Después, mostraremos su contenido para identificar las tablas disponibles antes de comenzar el procesamiento.


In [21]:

import pandas as pd
import numpy as np
import zipfile
from google.colab import files

# Cargar el archivo ZIP
archivos = files.upload()

nombre_zip = next(
    nombre for nombre in archivos
    if nombre.lower().endswith(".zip")
)

# Leer las tablas necesarias
with zipfile.ZipFile(nombre_zip, "r") as zip_ref:
    df_features = pd.read_csv(
        zip_ref.open("features_solicitud.csv")
    )

    df_solicitudes = pd.read_csv(
        zip_ref.open("solicitudes_prestamos.csv")
    )

print("Datos cargados correctamente")
print("Características:", df_features.shape)
print("Solicitudes:", df_solicitudes.shape)


Saving dataset_bbva_sintetico_v2.zip to dataset_bbva_sintetico_v2 (1).zip
Datos cargados correctamente
Características: (803, 24)
Solicitudes: (803, 16)


## Preparación del conjunto de datos
Ahora, seleccionamos únicamente los préstamos aprobados cuyo período de observación de $12$ meses ha finalizado y que cuentan con una etiqueta de incumplimiento conocida. Después, combinamos esas etiquetas con las características financieras de cada solicitud y eliminamos una variable redundante identificada durante nuestro análisis de correlaciones. Así obtenemos el conjunto de datos que utilizaremos para entrenar y evaluar el modelo.

In [22]:

# Variable objetivo
TARGET = "incumplimiento_90d_12m"

# Seleccionar préstamos con resultados conocidos
df_etiquetas = df_solicitudes[
    (df_solicitudes["decision_simulada"] == "APROBADA") &
    (df_solicitudes["etiqueta_madura"] == True) &
    (df_solicitudes[TARGET].notna())
][
    ["solicitud_id", "cliente_id",
     "particion_temporal", TARGET]
].copy()

# Integrar características y etiquetas
df_ml = df_features.merge(
    df_etiquetas,
    on=["solicitud_id", "cliente_id",
        "particion_temporal"],
    how="inner",
    validate="one_to_one"
)

assert len(df_ml) == len(df_etiquetas)

# Eliminar identificadores, metadatos y redundancia
columnas_excluir = [
    "solicitud_id",
    "cliente_id",
    "fecha_snapshot",
    "particion_temporal",
    "cuota_min_tarjeta_estimada_eur",
    TARGET
]

# Definir variables predictoras y objetivo
X = df_ml.drop(columns=columnas_excluir)
y = df_ml[TARGET].astype(int)

print("Observaciones:", len(X))
print("Variables predictoras:", X.shape[1])
print("Distribución del incumplimiento:")
print(y.value_counts())


Observaciones: 245
Variables predictoras: 19
Distribución del incumplimiento:
incumplimiento_90d_12m
0    216
1     29
Name: count, dtype: int64


## División de los datos
Bien, ahora dividimos las observaciones en entrenamiento, validación y prueba mediante las particiones temporales establecidas en el dataset original. Esta separación permite entrenar el modelo con un grupo de préstamos, evaluar su comportamiento con otro y reservar un tercer conjunto para la evaluación final.

In [23]:

# Identificar tipos de variables
variables_numericas = X.select_dtypes(
    include=["number"]
).columns.tolist()

variables_categoricas = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Particiones temporales
train_mask = df_ml["particion_temporal"] == "TRAIN"
val_mask = df_ml["particion_temporal"] == "VALIDACION"
test_mask = df_ml["particion_temporal"] == "TEST_TEMPORAL"

# Entrenamiento
X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

# Validación
X_val = X.loc[val_mask].copy()
y_val = y.loc[val_mask].copy()

# Prueba
X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

# Verificar las particiones
assert len(X_train) + len(X_val) + len(X_test) == len(X)

print("Entrenamiento:", X_train.shape)
print("Validación:", X_val.shape)
print("Prueba:", X_test.shape)


Entrenamiento: (119, 19)
Validación: (44, 19)
Prueba: (82, 19)


## Preprocesamiento y entrenamiento del modelo
Continuamos con la construcción de una regresión logística con regularización L2 y un parámetro $C=0.1$, configuración seleccionada tras los experimentos anteriores de los cuales no mostramos el proceso sino que lo mencionaremos en una memoria técnica.

Antes del entrenamiento, las variables numéricas se estandarizarán y las categóricas se convertirán en representaciones binarias mediante un pipeline de Scikit-learn.

El preprocesamiento aprende sus parámetros exclusivamente con los datos de entrenamiento para evitar fugas de información.

In [24]:

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Preprocesamiento numérico
pipeline_numerico = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler())
])

# Preprocesamiento categórico
pipeline_categorico = Pipeline([
    ("imputador", SimpleImputer(strategy="most_frequent")),
    ("codificador", OneHotEncoder(handle_unknown="ignore"))
])

# Combinar transformaciones
preprocesador = ColumnTransformer([
    ("numericas", pipeline_numerico, variables_numericas),
    ("categoricas", pipeline_categorico, variables_categoricas)
])

# Modelo de scoring crediticio
modelo_scoring = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", LogisticRegression(
        C=0.1,
        max_iter=2000,
        random_state=42
    ))
])

# Entrenar el modelo
modelo_scoring.fit(X_train, y_train)

print("Modelo entrenado correctamente")
print("Algoritmo: Regresión logística")
print("Regularización: L2")
print("C: 0.1")
print("Variables utilizadas:", X_train.shape[1])


Modelo entrenado correctamente
Algoritmo: Regresión logística
Regularización: L2
C: 0.1
Variables utilizadas: 19


## Evaluación del rendimiento predictivo
Bien, ahora vamos a evaluar el modelo utilizando las observaciones reservadas para validación y calculamos las métricas ROC-AUC, Average Precision y Brier Score. Estas medidas permiten analizar la capacidad de discriminación, la identificación de clientes de riesgo y la calidad de las probabilidades estimadas. Comparamos también el Brier Score con un modelo de referencia que asigna a todos los clientes la misma probabilidad de incumplimiento.

In [25]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Calcular probabilidades de incumplimiento
prob_val = modelo_scoring.predict_proba(X_val)[:, 1]

# Métricas predictivas
roc = roc_auc_score(y_val, prob_val)
ap = average_precision_score(y_val, prob_val)
brier = brier_score_loss(y_val, prob_val)

# Modelo de referencia
prob_base = np.full(len(y_val), y_train.mean())
brier_base = brier_score_loss(y_val, prob_base)

# Presentar resultados
resultados = pd.DataFrame({
    "Métrica": [
        "ROC-AUC",
        "Average Precision",
        "Brier Score",
        "Brier Score referencia"
    ],
    "Valor": [roc, ap, brier, brier_base]
})

print("EVALUACIÓN DEL MODELO DE SCORING")
display(resultados.round(4))


EVALUACIÓN DEL MODELO DE SCORING


,Métrica,Valor
0,ROC-AUC,0.7073
1,Average Precision,0.1746
2,Brier Score,0.0685
3,Brier Score referencia,0.0669


## Generación de probabilidades de riesgo por cliente
Ya que tenemos un modelo entrenado, utilicémoslo para calcular una probabilidad estimada de incumplimiento para cada solicitud del conjunto de validación. Organizamos los resultados en una tabla que incluye el identificador del cliente, su probabilidad estimada y el resultado real observado. Esta información constituye la base para desarrollar posteriormente un sistema de puntuación crediticia y segmentación de clientes por niveles de riesgo.

In [26]:

# Recuperar identificadores de validación
df_scoring = df_ml.loc[
    val_mask, ["cliente_id", "solicitud_id"]
].copy()

# Incorporar resultados
df_scoring["probabilidad_incumplimiento"] = prob_val
df_scoring["incumplimiento_real"] = y_val.to_numpy()

# Mostrar probabilidades como porcentaje
df_scoring["riesgo_estimado_pct"] = (
    df_scoring["probabilidad_incumplimiento"] * 100
).round(2)

# Ordenar clientes por riesgo estimado
df_scoring = df_scoring.sort_values(
    "probabilidad_incumplimiento",
    ascending=False
)

print("SCORING DE RIESGO CREDITICIO")
display(
    df_scoring[
        ["cliente_id", "solicitud_id",
         "riesgo_estimado_pct", "incumplimiento_real"]
    ].head(15)
)


SCORING DE RIESGO CREDITICIO


,cliente_id,solicitud_id,riesgo_estimado_pct,incumplimiento_real
64,CLI000261,SOL0000208,34.85,0
33,CLI000142,SOL0000111,33.27,0
119,CLI000505,SOL0000407,30.53,0
113,CLI000480,SOL0000388,27.99,1
56,CLI000233,SOL0000183,27.00,0
187,CLI000791,SOL0000636,26.54,0
220,CLI000909,SOL0000731,24.75,0
226,CLI000934,SOL0000751,22.28,0
58,CLI000240,SOL0000189,21.79,0
210,CLI000873,SOL0000703,19.86,0


## Interpretación de las variables del modelo.
Analizaremos ahora los coeficientes de la regresión logística para identificar qué características financieras están asociadas con un aumento o una disminución del riesgo de incumplimiento. Calcularemos también los odds ratios, que permiten expresar el efecto de cada variable sobre las probabilidades relativas de incumplimiento (odds). Este análisis nos ayudará a interpretar el comportamiento del modelo y comprobar si las relaciones aprendidas tienen sentido financiero.

In [27]:

import numpy as np
import pandas as pd

# 1. Recuperar los coeficientes del modelo
clasificador = modelo_scoring.named_steps["clasificador"]
preproceso = modelo_scoring.named_steps["preprocesamiento"]

coeficientes = clasificador.coef_[0]
nombres = preproceso.get_feature_names_out()

# 2. Crear tabla de coeficientes
df_coeficientes = pd.DataFrame({
    "Variable": nombres,
    "Coeficiente": coeficientes,
    "Odds Ratio": np.exp(coeficientes)
})

# 3. Identificar el tipo de variable
df_coeficientes["Tipo"] = np.where(
    df_coeficientes["Variable"].str.startswith("numericas__"),
    "Numérica",
    "Categórica"
)

# 4. Limpiar nombres
df_coeficientes["Variable"] = (
    df_coeficientes["Variable"]
    .str.replace(r"^(numericas|categoricas)__", "", regex=True)
)

# 5. Mostrar variables numéricas ordenadas
print("EFECTO DE LAS VARIABLES NUMÉRICAS")
print("=" * 60)

df_numericas = df_coeficientes[
    df_coeficientes["Tipo"] == "Numérica"
].sort_values("Coeficiente", ascending=False)

display(df_numericas.round(4))

# 6. Mostrar variables categóricas
print("\nCOEFICIENTES DE VARIABLES CATEGÓRICAS")
print("=" * 60)

df_categoricas = df_coeficientes[
    df_coeficientes["Tipo"] == "Categórica"
].sort_values("Coeficiente", ascending=False)

display(df_categoricas.round(4))


EFECTO DE LAS VARIABLES NUMÉRICAS


,Variable,Coeficiente,Odds Ratio,Tipo
2,variabilidad_ingresos_6m,0.2907,1.3374,Numérica
8,mora_previa_max_dias,0.2655,1.3040,Numérica
6,utilizacion_credito_previa,0.2516,1.2861,Numérica
9,dti_con_nueva_cuota,0.2009,1.2225,Numérica
5,credito_utilizado_ultimo_mes_eur,0.1219,1.1296,Numérica
4,limite_credito_total_ultimo_mes_eur,0.0985,1.1035,Numérica
17,tasa_nominal_anual_simulada,0.0332,1.0338,Numérica
14,importe_solicitado_eur,0.0281,1.0285,Numérica
15,plazo_meses,0.0156,1.0157,Numérica
11,accesos_digitales_6m,0.0107,1.0108,Numérica



COEFICIENTES DE VARIABLES CATEGÓRICAS


,Variable,Coeficiente,Odds Ratio,Tipo
19,situacion_laboral_AUTONOMO,0.1502,1.1621,Categórica
21,situacion_laboral_TEMPORAL,0.0156,1.0157,Categórica
20,situacion_laboral_PENSIONISTA,-0.0785,0.9245,Categórica
18,situacion_laboral_ASALARIADO,-0.0879,0.9159,Categórica


Bien, veamos nuestros resultados:
- Variabilidad de ingresos: ingresos menos estables se asocian con mayores probabilidades relativas de incumplimiento.
- Mora previa: los antecedentes de retrasos en pagos aparecen relacionados con mayor riesgo.
- Utilización del crédito: una mayor proporción de crédito utilizado puede reflejar presión financiera.
- Endeudamiento: destinar una mayor proporción de los ingresos al pago de deudas puede reducir la capacidad de afrontar nuevas obligaciones.

Por ejemplo, el odds ratio de $1,3374$ para la variabilidad de ingresos indica un incremento aproximado del $33,74 \%$ en los odds de incumplimiento por cada desviación estándar adicional, manteniendo constantes las demás variables.

## Descubriendo y eliminando variables constantes
Eliminaremos la variable constante para reducir de $19$ a $18$ las características predictoras y simplificar el modelo. Después entrenaremos nuevamente la regresión logística con la misma configuración y verificaremos que sus métricas de validación permanecen estables. Mantendremos intacto el conjunto de prueba temporal para reservarlo para la evaluación final.

In [28]:

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Variable sin variabilidad
variable_constante = "meses_historia_disponibles_6m"

# Reducir los conjuntos de datos
X_train_final = X_train.drop(columns=[variable_constante])
X_val_final = X_val.drop(columns=[variable_constante])
X_test_final = X_test.drop(columns=[variable_constante])

# Actualizar variables numéricas
numericas_final = [
    col for col in variables_numericas
    if col != variable_constante
]

# Construir el modelo simplificado
modelo_final = Pipeline([
    ("preprocesamiento", ColumnTransformer([
        ("numericas", clone(pipeline_numerico), numericas_final),
        ("categoricas", clone(pipeline_categorico),
         variables_categoricas)
    ])),
    ("clasificador", LogisticRegression(
        C=0.1,
        penalty="l2",
        max_iter=2000,
        random_state=42
    ))
])

# Entrenar el modelo
modelo_final.fit(X_train_final, y_train)

# Evaluar en validación
prob_final = modelo_final.predict_proba(X_val_final)[:, 1]

print("MODELO SIMPLIFICADO")
print("=" * 50)
print("Variables:", X_train_final.shape[1])
print(f"ROC-AUC: {roc_auc_score(y_val, prob_final):.4f}")
print(f"Average Precision: {average_precision_score(y_val, prob_final):.4f}")
print(f"Brier Score: {brier_score_loss(y_val, prob_final):.4f}")


MODELO SIMPLIFICADO
Variables: 18
ROC-AUC: 0.7073
Average Precision: 0.1746
Brier Score: 0.0685


Ahora vamos a transformar las probabilidades de incumplimiento estimadas por nuestro modelo en una puntuación crediticia en el rango de $300$ y $850$ puntos, donde las puntuaciones más altas indican menor riesgo.

Esta escala facilita la interpretación de los resultados y posteriormente nos permitirá clasificar a los clientes en diferentes niveles de riesgo. Utilizaremos una transformación logarítmica de los odds de cumplimiento frente a incumplimiento:

$$
\text { Score }=600+\frac{50}{\ln (2)} \ln \left(\frac{1-p}{9 p}\right);
$$

Donde $p$ es la probabilidad de incumplimiento estimada.


Hemos definido dos parámetros ilustrativos: $600$ puntos para una probabilidad de incumplimiento del $10 \%$ y $50$ puntos adicionales cada vez que se duplican los odds de cumplimiento frente a incumplimiento.

Por lo tanto, los clientes con mayores probabilidades estimadas de incumplimiento deberán obtener puntuaciones más bajas, y aquellos con menor riesgo estimado tendrán puntuaciones más altas.


In [29]:

import numpy as np
import pandas as pd

# 1. Parámetros de la escala crediticia
SCORE_BASE = 600
PD_BASE = 0.10
PDO = 50

# 2. Función de transformación de probabilidades
def calcular_score(probabilidades):

    p = np.clip(
        np.asarray(probabilidades, dtype=float),
        1e-6,
        1 - 1e-6
    )

    odds_base = (1 - PD_BASE) / PD_BASE
    odds_cliente = (1 - p) / p

    score = (
        SCORE_BASE
        + (PDO / np.log(2))
        * np.log(odds_cliente / odds_base)
    )

    return np.clip(
        np.rint(score), 300, 850
    ).astype(int)

# 3. Crear tabla de scoring para validación
df_scoring_final = df_ml.loc[
    val_mask,
    ["cliente_id", "solicitud_id"]
].copy()

df_scoring_final["probabilidad_impago"] = prob_final

df_scoring_final["score_crediticio"] = calcular_score(
    prob_final
)

df_scoring_final["incumplimiento_real"] = y_val.to_numpy()

# 4. Presentar probabilidades en porcentaje
df_scoring_final["riesgo_pct"] = (
    df_scoring_final["probabilidad_impago"] * 100
).round(2)

# 5. Ordenar de menor a mayor puntuación
df_scoring_final = df_scoring_final.sort_values(
    "score_crediticio"
)

print("SCORING DE RIESGO CREDITICIO")
print("=" * 60)

display(
    df_scoring_final[
        [
            "cliente_id",
            "riesgo_pct",
            "score_crediticio",
            "incumplimiento_real"
        ]
    ].head(15)
)

# 6. Resumen estadístico
print("\nDISTRIBUCIÓN DEL SCORE")
display(
    df_scoring_final["score_crediticio"].describe()
)


SCORING DE RIESGO CREDITICIO


,cliente_id,riesgo_pct,score_crediticio,incumplimiento_real
64,CLI000261,34.85,487,0
33,CLI000142,33.27,492,0
119,CLI000505,30.53,501,0
113,CLI000480,27.99,510,1
56,CLI000233,27.00,513,0
187,CLI000791,26.54,515,0
220,CLI000909,24.75,522,0
226,CLI000934,22.28,532,0
58,CLI000240,21.79,534,0
210,CLI000873,19.86,542,0



DISTRIBUCIÓN DEL SCORE


,score_crediticio
count,44.000000
mean,590.795455
std,56.372754
min,487.000000
25%,546.250000
50%,599.500000
75%,634.250000
max,716.000000


## Evaluación del modelo predictivo
Ahora, evaluaremos el modelo utilizando los $82$ préstamos del conjunto de prueba temporal, que no han participado en su entrenamiento ni en la selección de hiperparámetros. Calcularemos las métricas de discriminación y calidad probabilística, comparándolas con un modelo de referencia basado en la tasa de incumplimiento del entrenamiento. Esta evaluación nos permitirá conocer el rendimiento del modelo sobre observaciones reservadas y establecer las conclusiones finales del experimento.

In [32]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# 1. Obtener probabilidades en el conjunto de prueba
prob_test = modelo_final.predict_proba(
    X_test_final
)[:, 1]

# 2. Evaluar discriminación y probabilidades
roc = roc_auc_score(y_test, prob_test)
ap = average_precision_score(y_test, prob_test)
brier = brier_score_loss(y_test, prob_test)

# 3. Modelo de referencia
prob_base = np.full(len(y_test), y_train.mean())
brier_base = brier_score_loss(y_test, prob_base)

# 4. Comparar métricas
resultados_test = pd.DataFrame({
    "Métrica": [
        "ROC-AUC",
        "Average Precision",
        "Brier Score",
        "Brier Score referencia",
        "Tasa real de incumplimiento"
    ],
    "Resultado": [
        roc,
        ap,
        brier,
        brier_base,
        y_test.mean()
    ]
})

print("EVALUACIÓN FINAL DEL MODELO")
print("=" * 60)

display(resultados_test.round(4))

# 5. Evaluación con umbral exploratorio fijado en validación
umbral = 0.15

pred_test = (prob_test >= umbral).astype(int)

print(f"\nCLASIFICACIÓN CON UMBRAL = {umbral}")
print(f"Precision: {precision_score(y_test, pred_test, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, pred_test, zero_division=0):.4f}")
print(f"F1-score: {f1_score(y_test, pred_test, zero_division=0):.4f}")

# 6. Matriz de confusión
matriz = confusion_matrix(y_test, pred_test, labels=[0, 1])

df_matriz = pd.DataFrame(
    matriz,
    index=["Real: No incumple", "Real: Incumple"],
    columns=["Predicho: No incumple", "Predicho: Incumple"]
)

print("\nMATRIZ DE CONFUSIÓN")
display(df_matriz)


EVALUACIÓN FINAL DEL MODELO


,Métrica,Resultado
0,ROC-AUC,0.6402
1,Average Precision,0.3369
2,Brier Score,0.1077
3,Brier Score referencia,0.1162
4,Tasa real de incumplimiento,0.1341



CLASIFICACIÓN CON UMBRAL = 0.15
Precision: 0.2727
Recall: 0.5455
F1-score: 0.3636

MATRIZ DE CONFUSIÓN


,Predicho: No incumple,Predicho: Incumple
Real: No incumple,55,16
Real: Incumple,5,6


Bien, ya tenemos los resultados del modelo sobre los $82$ préstamos reservados para prueba. El ROC-AUC disminuye respecto a validación (de $0,7073$ a $0,6402$). Esto sugiere una capacidad de discriminación menor en prueba. Por otra parte, el Average Precision de $0,3369$ supera la prevalencia de incumplimiento de $0,1341$, lo que indica cierta utilidad para priorizar clientes de mayor riesgo. El Brier Score mejora aproximadamente un $7,3 \%$ respecto al modelo constante de referencia. Para un umbral exploratorio de $0,15$, el modelo genera $22$ alertas, de las cuales $6$ corresponden a incumplimientos reales. Esto representa una precisión del $27,27 \%$ y un recall del $54,55 \%$.

## Guardar el modelo entrenado
Ahora vamos a guardar el modelo de regresión logística junto con todas las transformaciones de preprocesamiento necesarias para reutilizarlo sin repetir el entrenamiento. También generaremos un archivo de metadatos que registre sus parámetros, las $18$ variables predictoras y la configuración de la escala crediticia. Estos archivos nos servirán posteriormente para implementar la arquitectura MLOps con Docker, Git-Flow y CI/CD.

In [33]:

import joblib
import json
import sklearn
from google.colab import files

# 1. Guardar el pipeline entrenado
nombre_modelo = "modelo_scoring_crediticio_v1.joblib"

joblib.dump(modelo_final, nombre_modelo)

# 2. Crear metadatos del modelo
metadatos = {
    "nombre": "Scoring de Riesgo Crediticio",
    "version": "1.0",
    "algoritmo": "LogisticRegression",
    "regularizacion": "L2",
    "C": 0.1,
    "variable_objetivo": TARGET,
    "variables_predictoras": X_train_final.columns.tolist(),
    "total_variables": X_train_final.shape[1],
    "version_sklearn": sklearn.__version__,
    "escala_scoring": {
        "minimo": 300,
        "maximo": 850,
        "score_base": 600,
        "pd_base": 0.10,
        "pdo": 50
    },
    "tipo_evaluacion": "Retrospectiva sobre datos sintéticos",
    "estado": "Prototipo experimental"
}

# 3. Guardar metadatos
nombre_metadatos = "metadatos_modelo_v1.json"

with open(nombre_metadatos, "w", encoding="utf-8") as f:
    json.dump(
        metadatos,
        f,
        indent=4,
        ensure_ascii=False
    )

print("MODELO GUARDADO CORRECTAMENTE")
print("Archivo del modelo:", nombre_modelo)
print("Archivo de metadatos:", nombre_metadatos)

# 4. Descargar archivos
files.download(nombre_modelo)
files.download(nombre_metadatos)


MODELO GUARDADO CORRECTAMENTE
Archivo del modelo: modelo_scoring_crediticio_v1.joblib
Archivo de metadatos: metadatos_modelo_v1.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>